## **Assignment 2 Group 63**
**Student Name:** Huzaifa Nissare-Houssen   
 **Student Number:** 300172186  
 **Student Name:** Fahmi Sajid Ahmed   
 **Student Number:** 300250180



### **Dataset Description: Netflix Movies and TV Shows**

---

#### **1. General Information**

 **Dataset Name:** Netflix Movies and TV Shows
   **Author:** Shivam Bansal   
   **Source:** https://www.kaggle.com/datasets/shivamb/netflix-shows  
   **Purpose:** This dataset was created to provide a comprehensive listing of all the movies and TV shows available on Netflix as of 2021. It is a real-world dataset intended for exploratory data analysis (EDA), trend tracking in the entertainment industry, and building recommendation engines. Researchers use it to compare content distribution across different countries and genres.

---

#### **2. Shape**

 **Rows:** 8,807  
 **Columns:** 12

---

#### **3. Features & Descriptions**

| Feature | Description | Type |
| --- | --- | --- |
| **`show_id`** | A unique identifier assigned to each movie or TV show. | **Categorical (ID)** |
| **`type`** | Identifies if the entry is a 'Movie' or a 'TV Show'. | **Categorical** |
| **`title`** | The official name of the content. | **Categorical (Text)** |
| **`director`** | The director(s) of the movie or show. (Contains many null values). | **Categorical** |
| **`cast`** | The primary actors and actresses featured in the title. | **Categorical** |
| **`country`** | The country or countries involved in the production of the content. | **Categorical** |
| **`date_added`** | The date the title was officially added to the Netflix library. | **Categorical (Date)** |
| **`release_year`** | The actual year the movie or show was first released. | **Numerical** |
| **`rating`** | The age-suitability rating (e.g., TV-MA, PG-13, R). | **Categorical** |
| **`duration`** | The length in minutes (Movies) or number of seasons (TV Shows). | **Numerical/Mixed** |
| **`listed_in`** | The genres assigned to the content (e.g., Horror, Documentaries). | **Categorical** |
| **`description`** | A short summary of the plot or premise. | **Categorical (Text)** |

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import random
import string

url = 'https://raw.githubusercontent.com/Fahmi-IT/CSI4142_A2/refs/heads/main/data/netflix_titles.csv'
clean_data = pd.read_csv(url)

clean_data.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [ ]:


def inject_numbers(df, column, decimal_percentage, min_val, max_val, type='int'):

    df_corrupted = df.copy()
    
    # 2. Calculate the number of rows to change
    n_rows = len(df_corrupted)
    n_to_replace = int(n_rows * decimal_percentage)
    
    if n_to_replace > 0:
        indices_to_replace = np.random.choice(df_corrupted.index, size=n_to_replace, replace=False)
        
        
        random_values = np.random.randint(min_val, max_val + 1, size=n_to_replace)
        if type == 'float':
            random_values = np.random.uniform(min_val, max_val + 1, size=n_to_replace)
        
        
        df_corrupted.loc[indices_to_replace, column] = random_values
        
    return df_corrupted




def add_numeric_sort_key(df, source_column):

    df_result = df.copy()
    cleaned_series = df_result[source_column].astype(str).str.replace(r'[^0-9]', '', regex=True)
    cleaned_series = cleaned_series.replace('', pd.NA)
    df_result['sort_key'] = pd.to_numeric(cleaned_series, errors='coerce').astype('Int64')

    return df_result

def compare_heads(df1, df2, label1="Original", label2="Modified"):
    # This creates a side-by-side view using basic HTML/CSS
    html_str = f"""
    <div style="display: flex; gap: 50px;">
        <div>
            <h3>{label1}</h3>
            {df1.head(3).to_html()}
        </div>
        <div>
            <h3>{label2}</h3>
            {df2.head(3).to_html()}
        </div>
    </div>
    """
    display(HTML(html_str))

def get_column_differences(df1, df2, join_col, compare_col):
    """
    Joins two dataframes and returns a list of IDs where 
    the compare_col values are different.
    """
    # 1. Merge the dataframes
    merged = pd.merge(df1, df2, on=join_col, suffixes=('_df1', '_df2'))
    
    # 2. Identify the differences
    # Note: Use .ne() or != for comparison
    diff_mask = merged[f'{compare_col}_df1'] != merged[f'{compare_col}_df2']
    
    # 3. Extract the join_col (show_id) for those rows
    affected_ids = merged.loc[diff_mask, join_col].unique().tolist()
    
    return affected_ids

def generate_report(clean_data, dirty_data, column):
    potential_to_remove = ['description', 'cast', 'date_added', 'listed_in', 'sort_key']
    
    if column in potential_to_remove:
        potential_to_remove.remove(column)
    
    sorted_clean = add_numeric_sort_key(clean_data, 'show_id').sort_values(by='sort_key')
    sorted_dirty = add_numeric_sort_key(dirty_data, 'show_id').sort_values(by='sort_key')
    
    diff = get_column_differences(sorted_clean, sorted_dirty, 'show_id', column)
    sorted_clean = sorted_clean[sorted_clean['show_id'].isin(diff)].copy()
    sorted_dirty = sorted_dirty[sorted_dirty['show_id'].isin(diff)].copy()
    
    
    compare_heads(
        sorted_clean.drop(columns=potential_to_remove, errors='ignore'), 
        sorted_dirty.drop(columns=potential_to_remove, errors='ignore'),
        label1="Original",
        label2="Changed"
    )
    
    

### Data Type Validatity Checker

This test verfies that the data type of the enteries is of the expected type

In [100]:


def inject_type_errors(df, column, decimal_percentage=0.1, error_type='float'):
    df_result = df.copy()
    

    df_result[column] = df_result[column].astype(object)
    
    n_rows = len(df_result)
    n_to_replace = int(n_rows * decimal_percentage)
    
    if n_to_replace > 0:
        target_indices = np.random.choice(df_result.index, size=n_to_replace, replace=False)
        
        if error_type == 'string':
            bad_data = "INVALID_TYPE"
        elif error_type == 'float':
            bad_data = 999.99  
        else:
            bad_data = None
            
        df_result.loc[target_indices, column] = bad_data
        
    return df_result


dirty_data = inject_type_errors(clean_data, 'release_year', 0.1, 'float')



In [101]:
def check_data_type(df, column, expected_type):
    errors = df[df[column].apply(lambda x: not isinstance(x, expected_type))]
    return errors

errors = check_data_type(dirty_data, 'release_year', int)



In [13]:
compare_heads(clean_data, errors, 'Original', 'Changed')

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war."
,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable."
5,s6,TV Show,Midnight Mass,Mike Flanagan,"Kate Siegel, Zach Gilford, Hamish Linklater, Henry Thomas, Kristin Lehman, Samantha Sloyan, Igby Rigney, Rahul Kohli, Annarah Cymone, Annabeth Gish, Alex Essoe, Rahul Abburi, Matt Biedel, Michael Trucco, Crystal Balint, Louis Oliver",NaN,"September 24, 2021",2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries","The arrival of a charismatic young priest brings glorious miracles, ominous mysteries and renewed religious fervor to a dying town desperate to believe."
8,s9,TV Show,The Great British Baking Show,Andy Devonshire,"Mel Giedroyc, Sue Perkins, Mary Berry, Paul Hollywood",United Kingdom,"September 24, 2021",2021,TV-14,9 Seasons,"British TV Shows, Reality TV","A talented batch of amateur bakers face off in a 10-week competition, whipping up their best dishes in the hopes of being named the U.K.'s best."


### Range Validity Checker

This test verifies if the data is in a valid range

In [73]:
def inject_numbers(df, column, decimal_percentage, min_val, max_val):

    df_corrupted = df.copy()
    
    # 2. Calculate the number of rows to change
    n_rows = len(df_corrupted)
    n_to_replace = int(n_rows * decimal_percentage)
    
    if n_to_replace > 0:
        indices_to_replace = np.random.choice(df_corrupted.index, size=n_to_replace, replace=False)
        
        random_values = np.random.randint(min_val, max_val + 1, size=n_to_replace)
        
        df_corrupted.loc[indices_to_replace, column] = random_values
        
    return df_corrupted

dirty_data = inject_numbers(clean_data,'release_year',0.1, 1765, 1890)

In [74]:
def check_range(df, column, min_val, max_val):
    return df[(df[column] < min_val) | (df[column] > max_val)]

errors = check_range(dirty_data, 'release_year', 1900, 2026)

In [75]:
#final report with example
generate_report(clean_data, errors, 'release_year')

[32, 36, 38, 42, 45, 53, 54, 58, 71, 78, 104, 114, 123, 144, 148, 157, 167, 177, 189, 203, 210, 225, 228, 231, 243, 270, 271, 301, 307, 316, 327, 328, 330, 346, 350, 353, 402, 411, 440, 450, 451, 458, 459, 463, 473, 479, 499, 500, 504, 509, 510, 513, 519, 539, 540, 541, 544, 560, 581, 583, 584, 590, 599, 605, 609, 635, 642, 650, 662, 670, 681, 696, 715, 718, 722, 728, 730, 752, 754, 776, 783, 785, 787, 813, 814, 815, 823, 826, 843, 876, 883, 886, 898, 917, 952, 955, 965, 969, 970, 974, 1004, 1008, 1039, 1045, 1046, 1053, 1076, 1090, 1103, 1108, 1109, 1118, 1120, 1137, 1138, 1140, 1153, 1155, 1160, 1162, 1183, 1184, 1188, 1201, 1206, 1207, 1218, 1222, 1229, 1266, 1275, 1289, 1303, 1315, 1328, 1339, 1346, 1348, 1363, 1376, 1384, 1396, 1401, 1422, 1433, 1464, 1465, 1471, 1488, 1503, 1516, 1526, 1528, 1542, 1544, 1550, 1552, 1558, 1571, 1574, 1576, 1577, 1580, 1585, 1594, 1601, 1602, 1621, 1629, 1631, 1648, 1657, 1675, 1678, 1687, 1701, 1710, 1711, 1715, 1716, 1720, 1721, 1746, 1747, 1748,

,show_id,type,title,director,country,release_year,rating,duration,listed_in
31,32,TV Show,Chicago Party Aunt,NaN,NaN,2021,TV-MA,1 Season,TV Comedies
35,36,Movie,The Father Who Moves Mountains,Daniel Sandu,NaN,2021,TV-MA,110 min,"Dramas, International Movies, Thrillers"
37,38,TV Show,Angry Birds,NaN,Finland,2018,TV-Y7,1 Season,"Kids' TV, TV Comedies"
,show_id,type,title,director,country,release_year,rating,duration,listed_in
31,32,TV Show,Chicago Party Aunt,NaN,NaN,1881,TV-MA,1 Season,TV Comedies
35,36,Movie,The Father Who Moves Mountains,Daniel Sandu,NaN,1880,TV-MA,110 min,"Dramas, International Movies, Thrillers"
37,38,TV Show,Angry Birds,NaN,Finland,1884,TV-Y7,1 Season,"Kids' TV, TV Comedies"


### Format Validity Checker

This test verifies if data is in a valid format

In [93]:
def inject_string_errors(df, column, percentage, string_length=5):
    """
    Creates a copy of a dataframe and replaces a % of a column 
    with a random string (e.g., 'aZ3pQ').
    """
    df_corrupted = df.copy()
    
    n_rows = len(df_corrupted)
    n_to_replace = int(n_rows * (percentage / 100))
    
    if n_to_replace > 0:
        indices_to_replace = np.random.choice(df_corrupted.index, size=n_to_replace, replace=False)
        
        chars = string.ascii_letters + string.digits
        
        random_strings = [
            ''.join(random.choice(chars) for _ in range(string_length)) 
            for _ in range(n_to_replace)
        ]
        
        df_corrupted.loc[indices_to_replace, column] = random_strings
        
    return df_corrupted

dirty_data = inject_string_errors(clean_data, 'date_added', 0.1)

In [94]:
def check_format(df, column, regex_pattern):
    return df[~df[column].astype(str).str.contains(regex_pattern, na=True)]


valid_pattern = '^(January|February|March|April|May|June|July|August|September|October|November|December)\\s\\d{1,2},\\s\\d{4}$'

errors = check_format(dirty_data, 'date_added', valid_pattern).head()

/var/folders/3w/1_sy9lz53_d08dv59ywyyw700000gn/T/ipykernel_46683/2485409835.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  return df[~df[column].astype(str).str.contains(regex_pattern, na=True)]


In [95]:
#final report with example
generate_report(clean_data, errors, 'date_added')

[738, 4180, 4861, 5349, 5495]


,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
737,738,TV Show,Trese,Jay Oliva,"Philippines, Singapore, Indonesia","June 11, 2021",2021,TV-MA,1 Season,"Anime Series, Crime TV Shows, TV Horror"
4179,4180,TV Show,Memory Love,NaN,NaN,"January 18, 2019",2017,TV-14,1 Season,"International TV Shows, Romantic TV Shows, TV Dramas"
4860,4861,Movie,Aiyaary,Neeraj Pandey,India,"May 15, 2018",2018,TV-MA,158 min,"Action & Adventure, Dramas, International Movies"
,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
737,738,TV Show,Trese,Jay Oliva,"Philippines, Singapore, Indonesia",eNAsi,2021,TV-MA,1 Season,"Anime Series, Crime TV Shows, TV Horror"
4179,4180,TV Show,Memory Love,NaN,NaN,8e26P,2017,TV-14,1 Season,"International TV Shows, Romantic TV Shows, TV Dramas"
4860,4861,Movie,Aiyaary,Neeraj Pandey,India,qP0yn,2018,TV-MA,158 min,"Action & Adventure, Dramas, International Movies"


### Consistency Validity Checker
 This test checks the consistency of a row

In [85]:

def invalidate_date_added(df, column='date_added', decimal_percentage=0.1):
  
    df_result = df.copy()
    

    n_rows = len(df_result)
    n_to_invalidate = int(n_rows * decimal_percentage)
    
    if n_to_invalidate > 0:
        target_indices = np.random.choice(df_result.index, size=n_to_invalidate, replace=False)
        
        
        subset = df_result.loc[target_indices, column].str.split(',', n=1, expand=True)
        
        if subset.shape[1] >= 2:
            numeric_part = pd.to_numeric(subset[1].str.strip(), errors='coerce')
            modified_number = (numeric_part - 100).astype(str)
            
       
            df_result.loc[target_indices, column] = subset[0] + ', ' + modified_number
            
    return df_result
dirty_data = invalidate_date_added(clean_data)

In [86]:
def year_consistency(df):
    df['date_added'] =  pd.to_numeric(df['date_added'].str.split(',', expand=True)[1], errors='coerce')
    mask = df['date_added'] < df['release_year']
    return df[mask]


errors = year_consistency(dirty_data)


In [87]:
#final report with example
generate_report(clean_data, errors, 'date_added')

[4, 27, 29, 37, 41, 64, 82, 88, 99, 103, 121, 123, 129, 161, 170, 177, 192, 215, 221, 230, 260, 264, 265, 266, 267, 270, 276, 322, 337, 372, 373, 376, 391, 401, 414, 421, 422, 430, 435, 455, 462, 470, 475, 479, 485, 491, 503, 511, 517, 520, 527, 531, 536, 560, 562, 571, 580, 591, 596, 599, 616, 649, 659, 671, 678, 679, 687, 698, 710, 711, 727, 728, 730, 739, 749, 755, 772, 778, 796, 827, 853, 866, 868, 874, 879, 896, 897, 921, 922, 923, 924, 960, 964, 965, 973, 979, 981, 985, 992, 1013, 1014, 1035, 1036, 1042, 1043, 1050, 1055, 1057, 1059, 1069, 1073, 1078, 1102, 1112, 1115, 1123, 1128, 1142, 1144, 1145, 1146, 1170, 1172, 1198, 1204, 1209, 1215, 1217, 1225, 1234, 1250, 1258, 1281, 1290, 1302, 1311, 1312, 1314, 1324, 1329, 1341, 1347, 1352, 1364, 1368, 1369, 1379, 1381, 1383, 1393, 1396, 1408, 1430, 1432, 1446, 1463, 1468, 1486, 1514, 1530, 1542, 1544, 1552, 1557, 1597, 1607, 1608, 1624, 1629, 1631, 1644, 1692, 1696, 1697, 1729, 1742, 1751, 1755, 1767, 1768, 1780, 1791, 1829, 1831, 1837

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
3,4,TV Show,Jailbirds New Orleans,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV"
26,27,Movie,Minsara Kanavu,Rajiv Menon,NaN,"September 21, 2021",1997,TV-PG,147 min,"Comedies, International Movies, Music & Musicals"
28,29,Movie,Dark Skies,Scott Stewart,United States,"September 19, 2021",2013,PG-13,97 min,"Horror Movies, Sci-Fi & Fantasy"
,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in
3,4,TV Show,Jailbirds New Orleans,NaN,NaN,1921.0,2021,TV-MA,1 Season,"Docuseries, Reality TV"
26,27,Movie,Minsara Kanavu,Rajiv Menon,NaN,1921.0,1997,TV-PG,147 min,"Comedies, International Movies, Music & Musicals"
28,29,Movie,Dark Skies,Scott Stewart,United States,1921.0,2013,PG-13,97 min,"Horror Movies, Sci-Fi & Fantasy"


### Uniqueness Validity Checker

This test validates that data is unique

In [16]:
def inject_ids(df, column, decimal_percentage, min_val, max_val):

    df_corrupted = df.copy()
    
    n_rows = len(df_corrupted)
    n_to_replace = int(n_rows * decimal_percentage)
    
    if n_to_replace > 0:
        indices_to_replace = np.random.choice(df_corrupted.index, size=n_to_replace, replace=False)
        
        random_values = np.random.randint(min_val, max_val + 1, size=n_to_replace)
        random_values_with_s = "s" + random_values.astype(str)
        
        df_corrupted.loc[indices_to_replace, column] = random_values_with_s
        
    return df_corrupted


dirty_data = inject_ids(clean_data, "show_id", 0.05, 1, 1000)

In [17]:
def check_uniqueness(df, column):
    return df[df.duplicated(subset=[column], keep=False)]


errors = check_uniqueness(dirty_data, 'show_id')
errors.head()


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, ...",NaN,"September 24, 2021",2021,PG,91 min,Children & Family Movies,Equestria's divided. But a bright-eyed hero be...
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra D...","United States, Ghana, Burkina Faso, United Kin...","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model s..."
12,s13,Movie,Je Suis Karl,Christian Schwochow,"Luna Wedler, Jannis Niewöhner, Milan Peschel, ...","Germany, Czech Republic","September 23, 2021",2021,TV-MA,127 min,"Dramas, International Movies",After most of her family is murdered in a terr...
14,s15,TV Show,Crime Stories: India Detectives,NaN,NaN,NaN,"September 22, 2021",2021,TV-MA,1 Season,"British TV Shows, Crime TV Shows, Docuseries",Cameras following Bengaluru police on the job ...


In [20]:

def get_common_rows(df1, df2, column):
    common_values = set(df2[column])
    common_df = df1[df1[column].isin(common_values)].copy()
    
    return common_df
    
    
common_values = get_common_rows(clean_data, errors, 'show_id')
compare_heads(
        common_values, 
       clean_data,
        label1="Original",
        label2="Changed"
    )

    

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war."
6,s7,Movie,My Little Pony: A New Generation,"Robert Cullen, José Luis Ucha","Vanessa Hudgens, Kimiko Glenn, James Marsden, Sofia Carson, Liza Koshy, Ken Jeong, Elizabeth Perkins, Jane Krakowski, Michael McKean, Phil LaMarr",NaN,"September 24, 2021",2021,PG,91 min,Children & Family Movies,"Equestria's divided. But a bright-eyed hero believes Earth Ponies, Pegasi and Unicorns should be pals — and, hoof to heart, she’s determined to prove it."
7,s8,Movie,Sankofa,Haile Gerima,"Kofi Ghanaba, Oyafunmike Ogunlano, Alexandra Duah, Nick Medley, Mutabaruka, Afemo Omilami, Reggie Carter, Mzuri","United States, Ghana, Burkina Faso, United Kingdom, Germany, Ethiopia","September 24, 2021",1993,TV-MA,125 min,"Dramas, Independent Movies, International Movies","On a photo shoot in Ghana, an American model slips back in time, becomes enslaved on a plantation and bears witness to the agony of her ancestral past."
,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabiha Akkari, Sofia Lesaffre, Salim Kechiouche, Noureddine Farihi, Geert Van Rampelberg, Bakary Diombera",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Action & Adventure","To protect his family from a powerful drug lord, skilled thief Mehdi and his expert team of robbers are pulled into a violent and deadly turf war."


### Presecence Validity Checker

This test validates data that is not null or empty

In [53]:
#introduce error
def inject_presence_errors(df, column, decimal_percentage):

    df_corrupted = df.copy()
    
    n_rows = len(df_corrupted)
    n_to_nullify = int(n_rows * decimal_percentage)
    
    if n_to_nullify > 0:
        indices_to_replace = np.random.choice(df_corrupted.index, size=n_to_nullify, replace=False)
        
     
        df_corrupted.loc[indices_to_replace, column] = np.nan
        
    return df_corrupted

dirty_data = inject_presence_errors(clean_data, 'description', 0.1)

In [54]:
def check_presence(df, column):
    return df[df[column].isna() | (df[column].astype(str).str.strip() == "")]

errors = check_presence(dirty_data, 'description')

In [ ]:
#final report
generate_report(clean_data, errors, 'description')

### Length Validity Checker

This test tests if data is a certian length

In [81]:
#introduce error
dirty_data = inject_string_errors(clean_data, 'director', 0.1, 50)

In [82]:
def check_length(df, column, min_len, max_len):
    lengths = df[column].astype(str).str.len()
    return df[(lengths < min_len) | (lengths > max_len)]

errors = check_length(dirty_data,'director', 1, 30)

In [ ]:
generate_report(clean_data, errors, 'director')

### Look-up Validity Checker
This test verifies that all that elements are part of a valid set


In [ ]:
dirty_data = inject_string_errors(clean_data, 'rating', 0.1)

In [ ]:

def get_invalid_tv_ratings(df, column='rating'):

  
    valid_ratings = {
        'TV-Y', 'TV-Y7', 'TV-G', 'TV-PG', 'TV-14', 'TV-MA', 
        'G', 'PG', 'PG-13', 'R', 'NC-17', 'NR', 'UR'
    }
    

    check_series = df[column].astype(str).str.strip().str.upper()
    

    invalid_mask = ~check_series.isin(valid_ratings)
    
    return df[invalid_mask].copy()

errors = get_invalid_tv_ratings(dirty_data)

In [ ]:
generate_report(clean_data, errors, 'ratings')